# 05 — The walk operator *is* attention: WalkAttention

**The paper's headline result.** The multi-hop walk operator that mitigates over-squashing is structurally an
*attention*: Transformer-style self-attention, but with the **support** given by the path algebra (which node
pairs are connected by length-`g` walks) instead of all-pairs, and with **learned** weights instead of fixed
multiplicity.

- `walkraw` weights every walk-reachable source *equally* (by multiplicity): it gets the signal through the
  bottleneck but cannot *select* which source matters.
- `WalkAttention` attends over the **same** range-`g` reachable pairs but **learns** query–key weights, so it
  both mitigates over-squashing *and* selects.
- 1-hop `GAT` cannot reach across the bottleneck at all.

The path-reachability mask is sparse (≈2% of all-pairs at the deepest range), so this is far cheaper than full
self-attention while staying global-in-reach. kQ/I's role is **constructive** — it defines the support — *not*
a destructive collapse (which failed; see notebook 04 / README).

Tracked in MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow.db`).

In [ ]:
import torch, torch.nn.functional as F, numpy as np, pandas as pd, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model

def ds(depth, n, seed, K=5, M=4):
    g = torch.Generator().manual_seed(seed + depth)
    return [make_bottleneck_graph(K, M, depth, g) for _ in range(n)]

## The sparse attention support

Confirm the walk-reachability mask is sparse and structured (the algebra defines *which* pairs attend).

In [ ]:
from oversquash.ideal_bridge import walk_operators
g = torch.Generator().manual_seed(0)
data0 = make_bottleneck_graph(5, 4, 3, g)
ei, N = data0.edge_index.numpy(), data0.x.size(0)
raw, _ = walk_operators(ei, N, max_length=4)
for gi in range(4):
    m = (raw[gi] > 0)
    print(f'range g={gi+1}: {m.sum():4d} reachable pairs / {N*N} dense ({100*m.sum()/(N*N):.0f}%)')

## Training harness (stabilized: LayerNorm in the model + warmup + grad-clip)

The LayerNorm lives in `WalkAttentionNet`; here we add the cosine-warmup schedule and gradient clipping that
make the learned attention converge reliably (without them, accuracy is high-ceiling but high-variance).

In [ ]:
def train_eval(m, tr, va, fwd, epochs=150, lr=2e-3, patience=30, warmup=10):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    def lr_lambda(ep):
        if ep < warmup: return (ep + 1) / warmup
        p = (ep - warmup) / max(1, epochs - warmup); return 0.5 * (1 + math.cos(math.pi * p))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best, bad = -1, 0
    def ev():
        m.eval(); acc = []
        with torch.no_grad():
            for b in va:
                lg, _ = fwd(b); mm = b.root_mask
                acc.append((lg[mm].argmax(-1) == b.y[mm]).float().mean().item())
        return float(np.mean(acc))
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg, _ = fwd(b)
            loss = F.cross_entropy(lg, b.y, ignore_index=-100); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        v = ev()
        if v > best: best, bad = v, 0
        else:
            bad += 1
            if bad >= patience: break
    return best

In [ ]:
rows = []
for depth in [2, 3]:
    for seed in [0, 1, 2, 3, 4]:
        nl = depth + 1
        for name in ['gat', 'walkraw', 'walkattn']:
            torch.manual_seed(seed)
            data = ds(depth, 3000, seed)
            if name == 'gat':
                tr = PyGLoader(data[:2400], batch_size=128, shuffle=True); va = PyGLoader(data[2400:], batch_size=128)
                fwd = lambda b: m(b.x, b.edge_index, getattr(b, 'batch', None))
            elif name == 'walkraw':
                tf = AttachWalkOperators(n_layers=nl); data = [tf(d) for d in data]
                tr = DataLoader(data[:2400], batch_size=128, shuffle=True, collate_fn=collate_walk_operators)
                va = DataLoader(data[2400:], batch_size=128, collate_fn=collate_walk_operators)
                fwd = lambda b: m(b.x, b.edge_index, getattr(b, 'batch', None), walk_raw=b.walk_raw)
            else:
                tf = AttachWalkMasks(n_layers=nl); data = [tf(d) for d in data]
                tr = DataLoader(data[:2400], batch_size=128, shuffle=True, collate_fn=collate_walk_masks)
                va = DataLoader(data[2400:], batch_size=128, collate_fn=collate_walk_masks)
                fwd = lambda b: m(b.x, b.edge_index, getattr(b, 'batch', None), walk_masks=b.walk_masks)
            in_dim, out_dim = data[0].x.size(-1), data[0].num_classes
            m = build_model(name, in_dim, 16, out_dim, nl, heads=4)
            acc = train_eval(m, tr, va, fwd)
            rows.append({'depth': depth, 'seed': seed, 'model': name, 'val_acc': acc})
            print(f'[d={depth} s={seed}] {name:9s} acc={acc:.3f}')
df = pd.DataFrame(rows)
df.groupby(['depth','model'])['val_acc'].agg(['mean','std','min','max']).round(3)

**Expected (verified, 5 seeds, unified stabilized protocol — `results/tables/bottleneck_main_5seeds.csv`):** `WalkAttention` = 1.000 ± 0.000 at both depths; `walkraw` ≈ 0.50–0.62; `GAT` ≈ 0.29–0.45 with large seed variance, degrading toward chance with depth (GCN/GIN sit at exactly chance — provably; see the paper’s formal account). The learned attention over the path-algebra support is
what wins — the bridge between quiver path-counting and Transformer attention.